# Cosmos DB Live Parallel Container Migration (Fabric)

This notebook orchestrates **parallel live migration** of multiple Cosmos DB containers by reading a CSV configuration file and calling `03_CosmosDB_Container_Migration` for each container pair.

Converted from [CosmosDBLiveParallelContainerMigration.scala](https://github.com/Azure/azure-sdk-for-java/tree/main/sdk/cosmos/azure-cosmos-spark_3/Samples/DatabricksLiveContainerMigration).

**How it works:**
| Step | Description |
|------|-------------|
| 1 | Install Cosmos DB Spark Connector |
| 2 | Upload/read CSV with container migration definitions |
| 3 | Configure parallelism |
| 4 | Build DAG and execute parallel notebook runs via `notebookutils.notebook.runMultiple()` |
| 5 | Collect and display results |

**Key differences from Databricks version:**
- Uses `notebookutils.notebook.runMultiple()` instead of Scala `Future` threading
- CSV stored in Lakehouse Files instead of DBFS
- Checkpoints stored in Lakehouse Files area

## Step 1: Configure Spark Session with Cosmos DB Spark Connector

In [1]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "Cosmos_Migration"
    }
}

StatementMeta(, 63543cf3-a1a5-4b9d-a0f9-1784888e74a3, -1, Finished, Available, Finished, True)

In [ ]:
# JAR Loading: SKIPPED - Using Fabric Environment
# The Fabric Environment "CosmosDB_Migration_Env" already provides the JAR:
#   /usr/lib/library-manager/bin/libraries/scala/azure-cosmos-spark_3-5_2-12-4.41.0.jar
# NO need to download or add JAR - it's automatically available to all notebooks

# Verify the JAR is available from Environment
print("✅ Cosmos DB Spark Connector JAR verified from Fabric Environment")
print("   Path: /usr/lib/library-manager/bin/libraries/scala/azure-cosmos-spark_3-5_2-12-4.41.0.jar")


## Step 2: Read Migration Configuration CSV

The CSV file `cosmosDBLiveMigrationList.csv` should be uploaded to Lakehouse Files with these columns:

| Column | Description |
|--------|-------------|
| `cosmosSourceEndpoint` | Cosmos DB source account URI |
| `cosmosSourceMasterKey` | Cosmos DB source account primary key |
| `cosmosRegion` | Preferred region (e.g., `[East US]`) |
| `cosmosSourceDatabaseName` | Source database name |
| `cosmosSourceContainerName` | Source container name |
| `cosmosSourceContainerThroughputControl` | Throughput control % (e.g., `0.95`) |
| `cosmosTargetEndpoint` | Target DB target account URL |
| `cosmosTargetMasterKey` | Target DB target account primary key |
| `cosmosTargetDatabaseName` | Target database name |
| `cosmosTargetContainerName` | Target container name |
| `cosmosTargetContainerPartitionKey` | Target partition key (e.g., `/pk`) |
| `cosmosTargetContainerProvisionedThroughput` | Target autoscale throughput (e.g., `10000`) |

> **Upload the CSV** to: `Files/cosmosDBLiveMigrationList.csv` in the `Cosmos_Migration` Lakehouse.

In [ ]:
# Read the migration configuration CSV from Lakehouse Files
csv_path = "Files/cosmosDBLiveMigrationList.csv"

migration_df = spark.read.option("header", True).csv(csv_path)

# Display available migrations
print(f"Found {migration_df.count()} container migration(s) defined in CSV:")
migration_df.show(truncate=False)

# Collect as list of dicts for building DAG activities
migration_list = [row.asDict() for row in migration_df.collect()]

## Step 3: Configure Parallelism

Set the maximum number of containers to migrate in parallel. Too many parallel notebooks may strain compute resources since they share the Spark session.

> **Recommendation:** Start with 4 and adjust based on cluster capacity.

In [ ]:
# Maximum number of containers to migrate in parallel
num_notebooks_in_parallel = 4

# Name of the child notebook to call for each container migration
notebook_to_run = "03_CosmosDB_Container_Migration"

print(f"Will run up to {num_notebooks_in_parallel} migrations in parallel")
print(f"Child notebook: {notebook_to_run}")
print(f"Total containers to migrate: {len(migration_list)}")

## Step 4: Build DAG & Execute Parallel Migrations

Builds a DAG of parallel activities from the CSV rows and executes them using `notebookutils.notebook.runMultiple()`. Each activity calls `03_CosmosDB_Container_Migration` with the parameters for that container pair.

> **Note:** All activities run independently (no dependencies between containers). The `concurrency` parameter controls how many run simultaneously.

In [ ]:
# Build DAG activities from CSV migration list
activities = []
for i, row in enumerate(migration_list):
    # Map CSV column names to notebook parameter names
    activity = {
        "name": f"Migrate_{i}_{row['cosmosSourceDatabaseName']}_{row['cosmosSourceContainerName']}",
        "path": notebook_to_run,
        "timeoutPerCellInSeconds": 600,
        "retry": 1,
        "retryIntervalInSeconds": 30,
        "args": {
            "cosmos_source_endpoint": row["cosmosSourceEndpoint"],
            "cosmos_source_master_key": row["cosmosSourceMasterKey"],
            "cosmos_target_endpoint": row["cosmosTargetEndpoint"],
            "cosmos_target_master_key": row["cosmosTargetMasterKey"],
            "cosmos_region": row["cosmosRegion"],
            "cosmos_source_database_name": row["cosmosSourceDatabaseName"],
            "cosmos_source_container_name": row["cosmosSourceContainerName"],
            "cosmos_source_throughput_control": row["cosmosSourceContainerThroughputControl"],
            "cosmos_target_database_name": row["cosmosTargetDatabaseName"],
            "cosmos_target_container_name": row["cosmosTargetContainerName"],
            "cosmos_target_container_partition_key": row["cosmosTargetContainerPartitionKey"],
            "cosmos_target_container_provisioned_throughput": row["cosmosTargetContainerProvisionedThroughput"],
        }
    }
    activities.append(activity)

# Build the DAG
dag = {
    "activities": activities,
    "timeoutInSeconds": 86400,  # 24 hours max
    "concurrency": num_notebooks_in_parallel
}

# Validate DAG
print(f"DAG built with {len(activities)} activities:")
for act in activities:
    print(f"  - {act['name']}")

is_valid = notebookutils.notebook.validateDAG(dag)
print(f"\nDAG validation: {'PASSED' if is_valid else 'FAILED'}")

## Step 5: Execute Parallel Migration

This cell launches all container migrations in parallel. It will block until all notebooks complete or timeout.

In [ ]:
# Execute all container migrations in parallel
print(f"Starting parallel migration of {len(activities)} containers (max {num_notebooks_in_parallel} concurrent)...")
print("=" * 70)

results = notebookutils.notebook.runMultiple(dag)

# Display results
print("\n" + "=" * 70)
print("MIGRATION RESULTS:")
print("=" * 70)

success_count = 0
error_count = 0
for activity_name, result in results.items():
    if result.get("exception"):
        print(f"  FAILED  - {activity_name}: {result['exception']}")
        error_count += 1
    else:
        print(f"  SUCCESS - {activity_name}: {result.get('exitVal', 'completed')}")
        success_count += 1

print(f"\nSummary: {success_count} succeeded, {error_count} failed out of {len(results)} total")